# Building a Transformer (GPT) from Scratch in PyTorch
Inspired by Andrej Karpathy's lecture: **[Let's build GPT: from scratch, in code, spelled out.](https://www.youtube.com/watch?v=kCc8FmEb1nY)**

In this notebook, we build an autoregressive character-level Generative Pre-trained Transformer (GPT) step-by-step from scratch:
1. **Dataset Exploration & Character-Level Tokenization**
2. **Train / Validation Splitting & Batching**
3. **Baseline: Bigram Language Model**
4. **The Mathematical Intuition of Self-Attention**
5. **Single Self-Attention Head (`Head`)**
6. **Multi-Head Attention (`MultiHeadAttention`)**
7. **Feed-Forward Network (`FeedForward`)**
8. **Transformer Block with Residual Connections & LayerNorm (`Block`)**
9. **Full GPT Architecture (`GPTLanguageModel`)**
10. **Training Loop & Text Generation**


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters
batch_size = 32      # How many independent sequences to process in parallel
block_size = 64      # Maximum context length for predictions
max_iters = 3000     # Training iterations
eval_interval = 300  # Evaluate loss every N steps
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200     # Number of batches to average loss over
n_embd = 128         # Embedding dimension
n_head = 4           # Number of attention heads (head_size = n_embd // n_head = 32)
n_layer = 4          # Number of Transformer blocks
dropout = 0.2        # Dropout probability

print(f"Using device: {device}")
torch.manual_seed(1337)


## Step 1: Dataset Exploration (Tiny Shakespeare)
We read `input.txt` containing 1,115,394 characters from Shakespeare's works.


In [ ]:
# Read dataset
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters in dataset: {len(text)}")
print("First 250 characters:\n", text[:250])


## Step 2: Character-Level Tokenization
We build a simple character-level tokenizer mapping each unique character to an integer (`stoi`) and vice-versa (`itos`).


In [ ]:
# Unique characters in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {''.join(chars)}")

# Character to integer and integer to character mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]         # String -> List[int]
decode = lambda l: ''.join([itos[i] for i in l]) # List[int] -> String

# Verify tokenizer
example_str = "To be or not to be"
encoded_example = encode(example_str)
print(f"Original: {example_str}")
print(f"Encoded:  {encoded_example}")
print(f"Decoded:  {decode(encoded_example)}")


## Step 3: Train / Validation Split & Batch Generation
We split the data into 90% training and 10% validation sets.
`get_batch` samples random chunks of length `block_size` across batch dimension `batch_size`.


In [ ]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print(f"Input batch shape:  {xb.shape}")
print(f"Target batch shape: {yb.shape}")


## Step 4: Loss Estimation Helper
Computes averaged train and validation loss without computing gradients.


In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


## Step 5: Self-Attention Mechanism (Single Head)
Attention allows tokens to look at prior tokens and aggregate information based on query-key affinities.

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V $$

- **Query ($Q$)**: What am I looking for?
- **Key ($K$)**: What do I contain?
- **Value ($V$)**: What information do I communicate if you attend to me?
- **Causal Mask (`tril`)**: Ensures future tokens are not visible to earlier tokens.


In [ ]:
class Head(nn.Module):
    """ One head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        
        # Compute attention affinities: (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        # Weighted aggregation of values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out


## Step 6: Multi-Head Attention
Rather than computing a single attention representation, we apply multiple attention heads in parallel and concatenate their outputs, followed by a linear projection.


In [ ]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


## Step 7: Feed-Forward Network
Once tokens have gathered information from each other (communication), each token processes this information individually through a feed-forward MLP (computation).


In [ ]:
class FeedForward(nn.Module):
    """ A simple linear layer followed by non-linearity and projection """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


## Step 8: Transformer Block (Residual Connections & LayerNorm)
A complete Transformer block stacks Multi-Head Attention and Feed-Forward layers with:
- **Residual skip-connections** ($x = x + \dots$) to facilitate stable gradient flow in deep networks.
- **Layer Normalization** (`nn.LayerNorm`) before each sub-layer (Pre-LN formulation).


In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


## Step 9: The Complete GPT Language Model
We assemble the full autoregressive Generative Pre-trained Transformer:
- Token and Position Embedding tables
- Stack of $N$ Transformer Blocks
- Final Layer Normalization
- Linear LM classification head projecting back to `vocab_size`
- Text generation method via iterative autoregressive sampling


In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Weight initialization matching Karpathy's lecture
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)
        x = self.blocks(x)     # (B, T, n_embd)
        x = self.ln_f(x)       # (B, T, n_embd)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Crop context to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # Focus on the last time step
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


## Step 10: Model Instantiation & Untrained Generation
Let's see what the untrained model produces (random gibberish).


In [ ]:
model = GPTLanguageModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params / 1e6:.2f} M")

# Generate before training
context = torch.zeros((1, 1), dtype=torch.long, device=device)
untrained_output = decode(model.generate(context, max_new_tokens=100)[0].tolist())
print("--- Untrained Model Output ---")
print(untrained_output)


## Step 11: Training the Transformer
We optimize using AdamW. The loss drops from ~4.2 towards ~1.5 - 1.8 as the model learns vocabulary, words, verse structure, character names, and punctuation.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"Step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


## Step 12: Generative Text Sampling
Now we sample text from our trained GPT model!


In [ ]:
print("--- Generated Text After Training ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
